In [ ]:
#|default_exp _helpers

In [ ]:
#|hide
from nblite import nbl_export; nbl_export();

In [ ]:
#|export
import json
import sys
import tomllib
from pathlib import Path
from typing import Annotated, Any, Optional

import typer

from netrun.net.config._net_config import NetConfig
from netrun.net.config._nodes import NodeConfig

# CLI Helpers

Shared utilities for the netrun CLI commands.

In [ ]:
#|export
ConfigOpt = Annotated[Optional[str], typer.Option("--config", "-c", help="Path to netrun config file.")]
PrettyOpt = Annotated[bool, typer.Option("--pretty/--compact", help="Pretty-print or compact JSON output.")]

In [ ]:
#|export
NETRUN_EXTENSIONS = (".netrun.json", ".netrun.toml")


def find_config(path: str | None) -> Path:
    """Find a netrun config file.

    If *path* is given, resolve and return it.
    Otherwise search the current directory for files ending with
    ``*.netrun.json`` or ``*.netrun.toml``.

    Raises:
        typer.Exit: If no config found, multiple found, or path doesn't exist.
    """
    if path is not None:
        p = Path(path).resolve()
        if not p.exists():
            typer.echo(f"Error: config file not found: {p}", err=True)
            raise typer.Exit(1)
        return p

    cwd = Path.cwd()
    matches = [
        f for f in cwd.iterdir()
        if f.is_file() and any(f.name.endswith(ext) for ext in NETRUN_EXTENSIONS)
    ]

    if len(matches) == 0:
        typer.echo("Error: no *netrun.json or *netrun.toml found in current directory.", err=True)
        raise typer.Exit(1)
    if len(matches) > 1:
        names = ", ".join(f.name for f in matches)
        typer.echo(f"Error: multiple config files found: {names}. Use --config to specify one.", err=True)
        raise typer.Exit(1)

    return matches[0].resolve()


def load_config(path: str | None) -> tuple[NetConfig, Path]:
    """Find and load a ``NetConfig`` from a file.

    Returns:
        Tuple of (config, resolved_file_path).
    """
    config_path = find_config(path)
    try:
        config = NetConfig.from_file(config_path)
    except Exception as e:
        typer.echo(f"Error loading config: {e}", err=True)
        raise typer.Exit(1)
    return config, config_path


def load_resolved_config(path: str | None) -> tuple[NetConfig, Path]:
    """Load and resolve a ``NetConfig`` (expanding factories).

    Temporarily adds ``project_root_path`` to ``sys.path`` so that
    dotted import paths relative to the project root (e.g.
    ``"nodes.double"``) can be resolved.

    Falls back to unresolved config if resolution still fails.

    Returns:
        Tuple of (config, resolved_file_path).
    """
    config, config_path = load_config(path)
    project_root = str(config.project_root_path)
    added = False
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        added = True
    try:
        config = config.resolve()
    except Exception as e:
        typer.echo(
            f"Warning: could not resolve config (factories may not be importable): {e}",
            err=True,
        )
    finally:
        if added:
            sys.path.remove(project_root)
    return config, config_path


def load_raw_data(path: str | None) -> tuple[dict[str, Any], Path]:
    """Find and load a config file as a raw dict.

    Needed for accessing top-level keys (like ``recipes``) that are
    not part of the ``NetConfig`` pydantic model.

    Returns:
        Tuple of (raw_dict, resolved_file_path).
    """
    config_path = find_config(path)
    try:
        content = config_path.read_text()
        suffix = config_path.suffix.lower()
        if suffix == ".json":
            data = json.loads(content)
        elif suffix == ".toml":
            data = tomllib.loads(content)
        else:
            typer.echo(f"Error: unsupported format {suffix}", err=True)
            raise typer.Exit(1)
    except (json.JSONDecodeError, tomllib.TOMLDecodeError) as e:
        typer.echo(f"Error parsing config: {e}", err=True)
        raise typer.Exit(1)
    return data, config_path


def output_json(data: Any, pretty: bool) -> None:
    """Print *data* as JSON to stdout."""
    indent = 2 if pretty else None
    typer.echo(json.dumps(data, indent=indent, default=str))


def get_node_by_name(config: NetConfig, name: str) -> NodeConfig:
    """Look up a node by name, exiting with code 1 if not found."""
    for n in config.graph.nodes:
        if n.name == name:
            if isinstance(n, NodeConfig):
                return n
    typer.echo(f"Error: node '{name}' not found.", err=True)
    raise typer.Exit(1)


def port_type_str(port_type: Any) -> str | None:
    """Convert a ``PortTypeSpec`` to a human-readable string."""
    if port_type is None:
        return None
    if isinstance(port_type, str):
        return port_type
    if isinstance(port_type, type):
        return port_type.__name__
    # PortTypeConfig
    if hasattr(port_type, "name"):
        return port_type.name
    return str(port_type)

In [ ]:
#|export
def write_config_data(data: dict, path: Path) -> None:
    """Write a raw config dict back to a file (.json or .toml).

    Warns on stderr when writing TOML (comments are lost on round-trip).
    """
    suffix = path.suffix.lower()
    if suffix == ".json":
        path.write_text(json.dumps(data, indent=2, default=str) + "\n")
    elif suffix == ".toml":
        try:
            import tomli_w
        except ImportError:
            typer.echo("Error: tomli-w is required for TOML output. Install with: pip install tomli-w", err=True)
            raise typer.Exit(1)
        typer.echo("Warning: writing TOML — any comments in the original file will be lost.", err=True)
        path.write_text(tomli_w.dumps(data))
    else:
        typer.echo(f"Error: unsupported config format '{suffix}'", err=True)
        raise typer.Exit(1)


def deep_merge(base: dict, override: dict) -> dict:
    """Recursively merge *override* into *base*, returning a new dict.

    If both values for a key are dicts, recurse. Otherwise *override* wins.
    Lists are replaced, not merged.
    """
    result = dict(base)
    for key, value in override.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = value
    return result


def auto_position(nodes: list[dict]) -> dict:
    """Compute a reasonable UI position for a new node.

    Scans existing ``extra.ui.position.x`` values and places the new node
    300px to the right of the rightmost node at the average y.
    """
    xs: list[float] = []
    ys: list[float] = []
    for n in nodes:
        pos = n.get("extra", {}).get("ui", {}).get("position", {})
        if "x" in pos:
            xs.append(float(pos["x"]))
        if "y" in pos:
            ys.append(float(pos["y"]))
    if xs:
        return {"x": max(xs) + 300, "y": sum(ys) / len(ys) if ys else 150}
    return {"x": 100, "y": 150}


def validate_after_write(path: Path) -> None:
    """Run validation on a config file after writing, printing warnings to stderr.

    Validation issues are informational only — the write already succeeded.
    """
    try:
        config = NetConfig.from_file(path)
        errors = config.graph.validate()
        for err in errors:
            typer.echo(f"Warning: {err.type}: {err.msg}", err=True)
    except Exception:
        pass  # File was already written; skip if validation itself fails